In [ ]:
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from scipy.special import gamma
from numba import jit
from regression_common import ridge_pair, lasso_coefficients, sparse_indices, discrepancy_candidate
from regression_results import BootstrapRun
from regression_spectral import SpectralDesign
from regression_bootstrap import bootstrap_errors, validate_backend
from threadpoolctl import threadpool_limits


In [ ]:
def compute_sigma(Y, X, beta):
    sigma = np.linalg.norm(Y - X @ beta)
    return sigma / np.sqrt(Y.shape[0])


In [ ]:
@jit(nopython=True)
def Nesterov(X, Y, t, k_max, delta, omega):
    beta_prev = np.zeros(X.shape[1])
    beta_cur = np.zeros(X.shape[1])
    k = 0
    for k in range(1, k_max + 1):
        # k counts gradient updates; the first gives beta_1 = t * X.T @ Y.
        momentum = 0.0 if k == 1 else (k - 2) / (k - 1 + omega)
        z_k = beta_cur + momentum * (beta_cur - beta_prev)
        beta_next = z_k + t * (X.T @ (Y - X @ z_k))
        beta_prev, beta_cur = beta_cur, beta_next
        if np.linalg.norm(Y - X @ beta_cur) <= delta:
            break
    return beta_cur, k

@jit(nopython=True)
def Nesterov_debias(X, Y, t, k_max, omega):
    beta_prev = np.zeros(X.shape[1])
    beta_cur = np.zeros(X.shape[1])
    for k in range(1, k_max + 1):
        momentum = 0.0 if k == 1 else (k - 2) / (k - 1 + omega)
        z_k = beta_cur + momentum * (beta_cur - beta_prev)
        beta_next = z_k + t * (X.T @ (Y - X @ z_k))
        beta_prev, beta_cur = beta_cur, beta_next
    return beta_cur


In [ ]:


def compute_norminf_Nesterov(seed):
    rng = np.random.default_rng(seed)
    ordinary_response = X @ beta_Nesterov + rng.normal(0, sigma_Nesterov, n)
    debiased_response = X @ beta_Nesterov_debias + rng.normal(0, sigma_Nesterov_debias, n)
    # Algorithm 1 keeps the original sample's selected parameter fixed.
    ordinary_star = Nesterov_debias(X, ordinary_response, t1, k_Nesterov, omega)
    debiased_first = Nesterov_debias(X, debiased_response, t1, k_Nesterov, omega)
    second_fit = Nesterov_debias(X, X @ debiased_first, t1, k_Nesterov, omega)
    debiased_star = 2 * debiased_first - second_fit
    ordinary_star[np.abs(ordinary_star) <= bn_Nesterov] = 0
    debiased_star[np.abs(debiased_star) <= bn_Nesterov_debias] = 0
    ordinary_error = np.linalg.norm(ordinary_star - beta_Nesterov, ord=np.inf)
    debiased_error = np.linalg.norm(debiased_star - beta_Nesterov_debias, ord=np.inf)
    return ordinary_error, debiased_error


In [ ]:
beta_names = [
     'beta_Nesterov', 'beta_Nesterov_debias',
]

variable_names = [
     'Nesterov', 'Nesterov_debias',
]


In [ ]:
n        = 1000
p        = 1000
I        = np.eye(p)
t1       = 2.5e-6
# t1       = 2.5e-6 100 95 3/5
# t1       = 3.5e-6 100 95 3
# t1       = 4e-6 100 100 3
max_iter = 10000
k_max    = 1000
omega    = 5


i_Nesterov = 0
i_Nesterov_debias = 0
Sigma_Nesterov = 0
Sigma_Nesterov_debias = 0
Inf_norm_Nesterov = 0
Inf_norm_Nesterov_debias = 0
Norm_Nesterov = 0
Norm_Nesterov_debias = 0

# Output controls do not change the estimators or parameter selection.
N = 1000
B = 500
confidence = 0.95
noise_sd = 2.0
seed_root = 20261006
output_root = 'outputs/regression-bootstrap-v2'
resume_from = None

# Execution controls only; statistical settings remain unchanged.
backend = 'cpu'
batch_size = 64
cpu_workers = 4
blas_threads = 8
gpu_devices = (0,)
backend_info = validate_backend(backend, gpu_devices=gpu_devices)


In [ ]:
# Keep the seeded covariance SVD independent of execution threads.
with threadpool_limits(limits=1):
    np.random.seed(24)
    cov_matrix = np.full((p, p), 0.5)
    np.fill_diagonal(cov_matrix, 2.0)

    mean_vector = np.zeros(p)
    X = np.random.multivariate_normal(mean_vector, cov_matrix, n)

    U, S, Vt = np.linalg.svd(X)

    beta = np.zeros(p)


    beta[0:5]   = 2.0   # βi = 2.0, i = 1, 2, 3, 4, 5
    beta[5:10]  = -2.0  # βi = -2.0, i = 6, 7, 8, 9, 10
    beta[10:15] = 1.0   # βi = 1.0, i = 11, 12, 13, 14, 15
    beta[15:20] = -1.0  # βi = -1.0, i = 16, 17, 18, 19, 20

    # Generate 20 random indices
    indices = sparse_indices(np.random, p, size=20)

    # Get the indices of non-zero elements in beta
    nonzero_indices = np.nonzero(beta)[0]
    random_beta = np.zeros(p)
    random_beta[indices] = beta[nonzero_indices]
    beta = random_beta
    nominal_nesterov_step = t1
    t1 = min(nominal_nesterov_step, 0.99 / S[0]**2)
    effective_nesterov_step = t1

    spectral_design = SpectralDesign(X, svd=(U, S, Vt))


In [ ]:
run_config = {
    'method': 'Nesterov', 'n': n, 'p': p, 'N': N, 'B': B,
    'sigma': noise_sd, 'confidence': confidence, 'seed_root': seed_root,
    'design_seed': 24, 'noise_model': 'normal',
    'largest_singular_value': float(S[0]),
    'execution': {'backend': backend, 'batch_size': batch_size,
                  'cpu_workers': cpu_workers, 'blas_threads': blas_threads,
                  'gpu_devices': list(gpu_devices)},
    'backend_info': backend_info,
    'preprocess_blas_threads': 1,
    'threshold_grid': [0.001, 0.001, 1.0],
    'threshold_percentile': 95,
    'method_settings': {'selection': 'TDP', 'step': t1, 'max_steps': max_iter, 'tau': 1.01, 'omega': omega, 'nominal_step': nominal_nesterov_step},
}
with threadpool_limits(limits=blas_threads), BootstrapRun(run_config, X, beta, 'BootstrapNesterov.ipynb',
                  output_root=output_root, resume_from=resume_from) as run:
    print('Run directory:', run.directory)
    for x in range(run.start_index, N):
        print(x)
        outer_rng = np.random.default_rng((seed_root, x, 0, 0))

        mean    = 0
        std_dev = noise_sd


        e = outer_rng.normal(mean, std_dev, n)
        norm_e = np.linalg.norm(e)
        Y = np.dot(X, beta) + e

        selection_settings = dict(run_config['method_settings'])
        delta = selection_settings['tau'] * norm_e
        selection_settings['delta'] = delta
        beta_Nesterov, beta_Nesterov_debias, selected_parameter, selection_diagnostics = spectral_design.select(
            Y, 'Nesterov', selection_settings, beta)
        k_Nesterov = int(selected_parameter)
        selected_residual_norm = selection_diagnostics['residual_norm']
        sigma_Nesterov = compute_sigma(Y, X, beta_Nesterov)
        sigma_Nesterov_debias = compute_sigma(Y, X, beta_Nesterov_debias)

        result = []

        for i in range(1, 1001):
            bn = 0.001 * i

            norms = [bn]
            for var_name in beta_names:
                var_copy = globals()[var_name].copy()
                var_copy[np.abs(var_copy) <= bn] = 0
                norm = np.linalg.norm(var_copy - beta)
                norms.append(norm)
            result.append(norms)
        columns = ['bn'] + [f'norm_{name}' for name in beta_names]
        df = pd.DataFrame(result, columns=columns)

        results = []


        for column in df.columns[1:]:
            min_val = df[column].min()
            min_indices = np.where(df[column] == min_val)[0]

            if len(min_indices) > 1 and np.all(np.diff(min_indices) == 1):

                # avg_index = np.mean(min_indices)
                avg_index = np.percentile(min_indices, 95)
            else:

                avg_index = min_indices[0]

            selected_bn = np.interp(avg_index, np.arange(len(df)), df['bn'].to_numpy())
            results.append([min_val, selected_bn])
            results_array = np.array(results)

        for i, name in enumerate(variable_names):
            globals()[f'bn_{name}'] = results_array[i, 1]

        for name in variable_names:
            beta_var = globals()[f'beta_{name}']
            bn_var = globals()[f'bn_{name}']
            beta_var[np.abs(beta_var) <= bn_var] = 0

        for name in variable_names:
            sigma = compute_sigma(Y, X, globals()[f'beta_{name}'])
            globals()[f'sigma_{name}'] = sigma

        Sigma_Nesterov += np.abs(sigma_Nesterov**2 - std_dev**2)
        Sigma_Nesterov_debias += np.abs(sigma_Nesterov_debias**2 - std_dev**2)
        Norm_Nesterov += np.linalg.norm(beta_Nesterov - beta)
        Norm_Nesterov_debias += np.linalg.norm(beta_Nesterov_debias - beta)

        norminf_Nesterov_list = []
        norminf_Nesterov_debias_list = []
        results = bootstrap_errors(
            spectral_design, 'Nesterov', selected_parameter, selection_settings,
            np.column_stack([beta_Nesterov, beta_Nesterov_debias]),
            [sigma_Nesterov, sigma_Nesterov_debias], [bn_Nesterov, bn_Nesterov_debias],
            B, seed_root, x, backend=backend, batch_size=batch_size,
            cpu_workers=cpu_workers, blas_threads=blas_threads, gpu_devices=gpu_devices,
        )
        for norminf_Nesterov, norminf_Nesterov_debias in results:
            norminf_Nesterov_list.append(norminf_Nesterov)
            norminf_Nesterov_debias_list.append(norminf_Nesterov_debias)
        inf_Nesterov = np.linalg.norm(beta - beta_Nesterov, ord=np.inf)
        inf_Nesterov_debias = np.linalg.norm(beta - beta_Nesterov_debias, ord=np.inf)
        Inf_norm_Nesterov += inf_Nesterov
        Inf_norm_Nesterov_debias += inf_Nesterov_debias
        if inf_Nesterov <= np.percentile(norminf_Nesterov_list, 100 * confidence):
            i_Nesterov += 1
        if inf_Nesterov_debias <= np.percentile(norminf_Nesterov_debias_list, 100 * confidence):
            i_Nesterov_debias += 1

        run.record(
            x, selected_parameter=selected_parameter,
            thresholds=[bn_Nesterov, bn_Nesterov_debias],
            estimates=np.column_stack([beta_Nesterov, beta_Nesterov_debias]),
            sigma=[sigma_Nesterov, sigma_Nesterov_debias],
            bootstrap_errors=np.asarray(results),
            threshold_scan=df.to_numpy(), threshold_columns=list(df.columns),
            diagnostics={**selection_diagnostics, 'parameter_kind': 'steps', 'selected_residual_norm': selected_residual_norm, 'max_steps': max_iter, 'at_iteration_cap': bool(k_Nesterov == max_iter), 'stopping_threshold': float(delta)},
        )


In [ ]:
# Read all committed replications, including those restored from a checkpoint.
summary = run.summary
i_Nesterov = int(round(summary['coverage_original'] * N))
i_Nesterov_debias = int(round(summary['coverage_debiased'] * N))
Sigma_Nesterov = summary['variance_absolute_error_original'] * N
Sigma_Nesterov_debias = summary['variance_absolute_error_debiased'] * N
Inf_norm_Nesterov = summary['linf_original'] * N
Inf_norm_Nesterov_debias = summary['linf_debiased'] * N
Norm_Nesterov = summary['l2_original'] * N
Norm_Nesterov_debias = summary['l2_debiased'] * N
print(pd.Series(summary, name='Nesterov'))
print('Saved results:', run.directory)
